# Qwen3.8 SFT サンプル（Unsloth）

> **注意: これは動作確認用のサンプル。**
> 学習データは、このノートブックに埋め込んだ**架空の一般的な社内ヘルプデスク FAQ 12 件**だけで、
> 学習後のモデルはその内容を**覚えさせた（ほぼ丸暗記した）状態**になっている。実在の企業・組織の規程やデータは含まない。
> 学習データにない社内規程を聞くと架空の数値を断定的に答える（8 章の簡易評価を参照）ので、実務の回答用途には使わないこと。

Unsloth 公式ノートブック [`nb/Qwen3_6_MoE.ipynb`](https://github.com/unslothai/notebooks/blob/main/nb/Qwen3_6_MoE.ipynb)
（LGPL-3.0）をベースに、日本語の小さな社内 FAQ データで SFT する例に書き換えたもの。
モデルは dense の `unsloth/Qwen3.8-27B`。

Qwen3.8 は Qwen3.6 と同じ `qwen3_5` アーキテクチャ（`Qwen3_5ForConditionalGeneration`）で、
config と重みのテンソル構成も Qwen3.6-27B と同一。そのためインストール環境は Qwen3.6 版とほぼ同じ（torchao の上限を追加しただけ）。
変わったのは**チャットテンプレート**（reasoning effort の指示文が system に入る、推論過程は `reasoning_content` から読む）で、
これに合わせてデータ整形を直している（5 章）。

## 必要な環境

| 構成 | 重み | 学習時のおおよその VRAM | 動く GPU |
|---|---|---|---|
| `unsloth/Qwen3.8-27B`（dense, bf16 LoRA） | 約 56GB | 60GB 前後 | H100 80GB / A100 80GB / RTX PRO 6000 96GB |

- 重みのサイズは safetensors 18 ファイルの合計（約 55.6GB）。
  学習時の値は LoRA・オプティマイザ・アクティベーションを加えた**概算**で、実測値ではない。
- **Colab の無料 T4（16GB）や標準の A100 40GB では動かない。** 80GB クラスの GPU を割り当てること。
- Unsloth は Qwen3.8 用に 4bit 版（`unsloth/Qwen3.8-27B-unsloth-bnb-4bit`）も出しているが、
  このノートブックは bf16 LoRA で組んでおり、4bit の経路は確認していない。
- `Qwen3.8-Flash-Next` は別アーキテクチャなので、このノートブックの対象外。
- 2026-10-03 に Colab の RTX PRO 6000 Blackwell（96GB）で、上から最後まで（W&B への記録と Hugging Face へのアップロードを含めて）通ることを確認した（30 ステップの学習は約 40 秒）。

## 1. インストール

このセルは公式ノートブックのインストールセルをそのまま使っている。
`transformers==5.3.0` と `trl==0.22.2` が必須で、これより古い transformers では Qwen3.5 系（Qwen3.6 / Qwen3.8）は読み込めない。

Unsloth 2026.7.5 は Qwen3.8 の公開（2026 年 8 月）より前の版だが、Unsloth は Qwen3.5 系の処理を
モデル名ではなく `model_type`（`qwen3_5`）で切り替えるので、Qwen3.8 もこのバージョンで扱える。

**torchao は 0.17 系までに固定している。** 2026 年 8 月公開の torchao 0.18.0 は torch 2.10 以降にしかない
`torch.nn.functional.ScalingType` を import するため、torch 2.8 の環境では `from unsloth import FastLanguageModel` が
`ImportError: cannot import name 'ScalingType'` で落ちる。

In [1]:
%%capture
import os, importlib.util
UNSLOTH_VERSION = "2026.7.5"
UNSLOTH_ZOO_VERSION = "2026.7.6"
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base]=={UNSLOTH_ZOO_VERSION}" \
        "unsloth[base]=={UNSLOTH_VERSION}"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq "unsloth=={UNSLOTH_VERSION}" "unsloth_zoo=={UNSLOTH_ZOO_VERSION}"
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 "unsloth=={UNSLOTH_VERSION}" "unsloth_zoo=={UNSLOTH_ZOO_VERSION}"
!uv pip install transformers==5.3.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
# torchao 0.18.0 (2026-08) は torch 2.10 以降の torch.nn.functional.ScalingType を import するため、
# torch 2.8 では `from unsloth import ...` が ImportError で落ちる。0.17 系までに固定する。
!uv pip install --no-deps "torchao>=0.16.0,<0.18.0"
from importlib.metadata import version
assert torch.__version__.startswith("2.8.0"), f"torch が古いままです: {torch.__version__}。ランタイムを再起動して上から再実行してください。"
assert version("transformers") == "5.3.0", version("transformers")
assert version("trl") == "0.22.2", version("trl")
assert version("unsloth") == UNSLOTH_VERSION, version("unsloth")
assert version("unsloth_zoo") == UNSLOTH_ZOO_VERSION, version("unsloth_zoo")
from packaging.version import Version
assert Version(version("torchao")) < Version("0.18.0"), f"torchao {version('torchao')} は torch 2.8 と非互換です"

Qwen3.8（Qwen3.5 系）はハイブリッド線形アテンションを使うため、`flash-linear-attention` と `causal_conv1d` の
ビルド済み wheel があると学習が速くなる。無くても学習自体は動く。

In [2]:
%%capture
import json, platform, sys, torch
from urllib.request import urlopen

py   = f"cp{sys.version_info.major}{sys.version_info.minor}"
tv   = ".".join(torch.__version__.split("+")[0].split(".")[:2])
# ビルド済み CUDA wheel は torch 2.10 まで。2.11 / 2.12 は ABI 互換なので 2.10 に寄せる。
tv   = {"2.11": "2.10", "2.12": "2.10"}.get(tv, tv)
assert (cu := (torch.version.cuda or "").split(".")[0]), "CUDA-enabled PyTorch required."
abi  = "TRUE" if torch.compiled_with_cxx11_abi() else "FALSE"
plat = "linux_x86_64" if platform.machine().lower() in ("x86_64", "amd64") else "linux_aarch64"
whl  = f"{py}-{py}-{plat}.whl"

def find(repo, tag, match):
    api = f"https://api.github.com/repos/{repo}/releases/tags/{tag}"
    return next((a["browser_download_url"] for a in json.load(urlopen(api))["assets"] if match(a["name"])), None)

cc1d = find("Dao-AILab/causal-conv1d", "v1.6.1.post4",
            lambda n: n.endswith(whl) and f"+cu{cu}torch{tv}" in n and f"cxx11abi{abi}" in n)
if not cc1d:
    # 致命的ではない。上のセルでソースからビルド済みだし、無くても学習は通る。
    print(f"No prebuilt causal-conv1d wheel for torch {torch.__version__}/cu{cu}/{py}/abi{abi}, skipping it.")
fla = find("fla-org/flash-linear-attention", "v0.4.2", lambda n: n.endswith(whl)) \
      or "https://github.com/fla-org/flash-linear-attention/archive/refs/tags/v0.4.2.tar.gz"
wheels = " ".join(f'"{w}"' for w in (cc1d, fla) if w)

!uv pip install -qqq --no-deps $wheels
!pip uninstall -y sentence-transformers torchcodec  # torchcodec の import が Colab で壊れているため

## 2. GPU の確認

**先にここを通すこと。** 55GB 超のモデルをダウンロードしてから VRAM 不足で落ちるのが一番痛い。

Colab の A100 には 40GB と 80GB があり、[ランタイム] > [ランタイムのタイプを変更] で
A100 を選んだうえで **「ハイメモリ」を有効にする**と 80GB になる。
40GB のままでは Qwen3.8-27B の LoRA 学習はできない。

> **セルは必ず上から順に実行すること。** インストールセルは Colab に最初から入っている
> torch を 2.8.0 に差し替える。それより先に `import torch` を実行してしまうと古い torch が
> カーネルに残り、`RuntimeError: operator torchvision::nms does not exist` で
> モデルの読み込みに失敗する。順番を崩したときは
> [ランタイム] > **[セッションを再起動してすべて実行する]** でやり直す。

In [3]:
import subprocess

MODEL_NAME = "unsloth/Qwen3.8-27B"  # Qwen3.8 の dense 27B（bf16）
IS_MOE     = "A3B" in MODEL_NAME

# bf16 の重み + LoRA + オプティマイザ + アクティベーションの概算
MIN_VRAM_GB = {"unsloth/Qwen3.8-27B": 60.0}.get(MODEL_NAME, 60.0)

# あえて torch を import せず nvidia-smi で調べる。
# インストールセルより先に torch を import してしまうと、Colab に元から入っている
# 古い torch がカーネルに居座り、後から入れた torch 2.8.0 用の torchvision と食い違って
# "RuntimeError: operator torchvision::nms does not exist" になる。
probe = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
    capture_output=True, text=True,
)
if probe.returncode != 0 or not probe.stdout.strip():
    raise RuntimeError(
        "GPU ランタイムが選択されていません。"
        "[ランタイム] > [ランタイムのタイプを変更] で A100 を選んでください。"
    )

gpu_name, vram_mib, compute_cap = [x.strip() for x in probe.stdout.strip().splitlines()[0].split(",")]
total_gb = int(vram_mib) / 1024
GPU_COMPUTE_CAP = float(compute_cap)   # 次のセルで MoE バックエンドの判定に使う
print(f"GPU        : {gpu_name}")
print(f"VRAM       : {total_gb:.1f} GB")
print(f"compute cap: {GPU_COMPUTE_CAP}  (A100 = 8.0 / H100 = 9.0)")
print(f"モデル     : {MODEL_NAME}")
print(f"必要 VRAM  : {MIN_VRAM_GB:.0f} GB 以上（概算）")

if total_gb < MIN_VRAM_GB:
    raise RuntimeError(
        f"VRAM が足りません（{total_gb:.1f} GB < {MIN_VRAM_GB:.0f} GB）。\n"
        "  - Colab の A100 は 40GB と 80GB があります。ランタイムのタイプを変更で\n"
        "    A100 を選び「ハイメモリ」を有効にすると 80GB になります。\n"
        "  - それでも足りない場合は max_seq_length を 1024 に下げてください。"
    )

if total_gb < MIN_VRAM_GB + 10:
    print(
        "\n[注意] VRAM に余裕がありません。OOM が出たら max_seq_length を 1024 に、"
        "それでも駄目なら lora_rank を 4 に下げてください。"
    )

GPU        : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM       : 95.6 GB
compute cap: 12.0  (A100 = 8.0 / H100 = 9.0)
モデル     : unsloth/Qwen3.8-27B
必要 VRAM  : 60 GB 以上（概算）


## 3. モデルの読み込み

Qwen3.8 は VLM アーキテクチャ（`Qwen3_5ForConditionalGeneration`）なので、
`from_pretrained` の 2 番目の戻り値は tokenizer ではなく **processor** になる。
テキストのトークナイズには `processor.tokenizer` を使う。

Qwen3.8-27B は dense なので `IS_MOE` は False になり、MoE 用の処理（バックエンド固定・`gate_up_proj` の追加）は自動的に飛ばされる。

In [4]:
import os

# PyTorch 2.8 の torch._grouped_mm は Blackwell (compute capability 10 以上) では
# 実行できない。Unsloth 2026.7.5 の unsloth_triton も MoE LoRA の一部で同演算へ
# 戻るため、この組み合わせだけ import 前に native_torch へ固定する。
if IS_MOE:
    requested_backend = os.environ.get("UNSLOTH_MOE_BACKEND")
    if requested_backend:
        print(f"明示された MoE バックエンドを使用: {requested_backend}")
    elif GPU_COMPUTE_CAP >= 10.0:
        os.environ["UNSLOTH_MOE_BACKEND"] = "native_torch"
        print(f"PyTorch 2.8 + Blackwell ({GPU_COMPUTE_CAP}) のため MoE バックエンドを native_torch に固定")
    else:
        print("MoE バックエンドは Unsloth の実機プローブで自動選択")

from unsloth import FastLanguageModel
import torch
if IS_MOE:
    from unsloth_zoo.temporary_patches.moe_utils import select_moe_backend
    MOE_BACKEND = select_moe_backend()
    print(f"選択された MoE バックエンド: {MOE_BACKEND}")

# MODEL_NAME と IS_MOE は前のセルで定義済み
max_seq_length = 2048  # 長い reasoning を学習するなら増やす。OOM 時は 1024 に下げる
lora_rank      = 8     # 大きいほど表現力は上がるが遅くなる

model, processor = FastLanguageModel.from_pretrained(
    MODEL_NAME,          # 巨大なのでダウンロードに時間がかかる
    max_seq_length = max_seq_length,
    load_in_4bit   = False,  # bf16 のまま LoRA（80GB 以上の GPU 前提）
    fast_inference = False,  # Qwen3.5 系（Qwen3.8 含む）では False
)
tokenizer = processor.tokenizer  # テキストのトークナイズ用

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.5: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

## 4. LoRA アダプタの追加

attention と MLP の射影に LoRA を当てる。Qwen3.8-27B は dense なので `gate_up_proj`（MoE 用）は追加されない。

In [5]:
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
if IS_MOE:
    target_modules.append("gate_up_proj")  # MoE 層にも LoRA を当てる

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,                  # 8, 16, 32, 64, 128 あたりから選ぶ
    target_modules = target_modules,
    lora_alpha = lora_rank * 2,     # rank の 2 倍にすると学習が速い
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,  # VRAM を節約
    random_state = 3407,
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


## 5. データの準備

`messages` 形式（OpenAI 互換）の会話をそのまま使う。ここでは動作確認用に
社内ヘルプデスク風の日本語サンプルを 12 件だけ埋め込んでいる。
実際に使うときは自前のデータに差し替えること（このサイズでは学習の効果は測れない）。

リポジトリ同梱の `data/sample_ja.jsonl` を読む場合は、下のセルの代わりに
`load_dataset("json", data_files="data/sample_ja.jsonl", split="train")` を使う。

In [6]:
from datasets import Dataset

SYSTEM = "あなたは社内ヘルプデスクのアシスタントです。日本語で、結論から簡潔に答えてください。"

QA_PAIRS = [
    ("経費精算の締め日を教えてください。",
     "毎月25日が締め日です。25日が土日祝の場合は直前の営業日に前倒しになります。締め後の申請は翌月分として処理されます。"),
    ("社給パソコンのパスワードを忘れました。どうすればいいですか。",
     "情報システム部への電話連絡で初期化できます。内線4120に本人から連絡し、社員番号を伝えてください。再設定は原則として当日中に完了します。"),
    ("有給休暇の残日数はどこで確認できますか。",
     "勤怠システムの「休暇管理」画面で確認できます。トップページ右上のメニューから開き、当年度付与分と繰越分が分けて表示されます。"),
    ("在宅勤務の申請は何日前までに出す必要がありますか。",
     "前営業日の17時までです。緊急時は当日申請も認められますが、事後に上長の承認が必要になります。"),
    ("会議室を予約したいのですが、方法がわかりません。",
     "グループウェアの「施設予約」から予約します。日時と人数を入力すると空いている会議室が一覧表示されるので、そこから選択してください。30分単位で最大4時間まで押さえられます。"),
    ("出張の航空券は自分で手配してよいですか。",
     "原則として指定の旅行代理店経由で手配します。自己手配は代理店で取り扱いがない路線に限り、事前に上長の承認を得た場合のみ認められます。"),
    ("健康診断はいつ受けられますか。",
     "毎年9月から11月にかけて実施されます。対象者には8月中旬に案内メールが届くので、記載の予約サイトから希望日を選んでください。"),
    ("退職した社員のファイル共有権限はどう扱われますか。",
     "退職日の翌営業日に自動で無効化されます。引き継ぎが必要なファイルは、退職日までに所属部門の共有フォルダへ移動しておいてください。"),
    ("社外の人とファイルをやり取りしたいのですが、メール添付でよいですか。",
     "10MBを超える場合は社内のファイル転送サービスを使ってください。10MB以下であればメール添付で問題ありませんが、機密区分が「社外秘」以上の資料はパスワード保護が必須です。"),
    ("残業時間の上限について教えてください。",
     "原則として月45時間、年間360時間が上限です。特別条項の適用時でも月100時間未満、複数月平均80時間以内に収める必要があります。"),
    ("備品の購入申請はどこから出しますか。",
     "購買システムの「一般備品申請」から出します。5万円未満は部門長承認のみ、5万円以上は部門長と経理部の二段階承認が必要です。"),
    ("VPNに接続できません。",
     "まず社給端末の証明書の有効期限を確認してください。期限切れが原因の場合が大半です。証明書が有効なのに接続できないときは、内線4120へ端末名とエラーコードを添えて連絡してください。"),
]

dataset = Dataset.from_list([
    {"messages": [
        {"role": "system",    "content": SYSTEM},
        {"role": "user",      "content": q},
        {"role": "assistant", "content": a},
    ]}
    for q, a in QA_PAIRS
])
print(dataset)

Dataset({
    features: ['messages'],
    num_rows: 12
})


### chat template の適用

`add_generation_prompt=False` で会話全体を 1 本のテキストにする。
Qwen3.8 のテンプレートは **assistant ターンに必ず thinking ブロックを差し込む**:

```
<|im_start|>assistant\n<think>\n{reasoning_content}\n</think>\n\n{content}<|im_end|>
```

`reasoning_content` を持たないメッセージでは reasoning が空になり、
推論時に `enable_thinking=False` で作るプロンプト
（`<|im_start|>assistant\n<think>\n\n</think>\n\n`）と一致する。

**Qwen3.6 からの変更点（ここを直さないと学習と推論でプロンプトがずれる）**

- thinking が有効（`enable_thinking` 未指定 or True）だと、system プロンプトの先頭に
  「Reasoning effort is set to xhigh. ...」という指示文が自動で入る（`reasoning_effort` の既定が `xhigh`）。
  非 thinking の SFT では、**学習データも `enable_thinking=False` で整形**してこの指示文を入れない。
- 推論過程は content 内の `<think>` タグからではなく、メッセージの `reasoning_content` フィールドから読まれる。
  thinking を学習させたい場合は assistant メッセージに `"reasoning_content": "（推論過程）"` を足し、
  下のセルの `enable_thinking = False` を外す（推論時も `enable_thinking=True` で揃える）。

In [7]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize = False,
            add_generation_prompt = False,
            # Qwen3.8: thinking 有効時は system に reasoning effort の指示文が挿入される。
            # 推論時（enable_thinking=False）とプロンプトを揃えるため、学習データも False で整形する。
            enable_thinking = False,
        )
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

In [8]:
print(dataset[0]["text"])

<|im_start|>system
あなたは社内ヘルプデスクのアシスタントです。日本語で、結論から簡潔に答えてください。<|im_end|>
<|im_start|>user
経費精算の締め日を教えてください。<|im_end|>
<|im_start|>assistant
<think>

</think>

毎月25日が締め日です。25日が土日祝の場合は直前の営業日に前倒しになります。締め後の申請は翌月分として処理されます。<|im_end|>



## 6. 学習

動作確認用に 30 ステップだけ回す。実際に使うときは `max_steps` を外して
`num_train_epochs = 1` などに置き換える。

学習の損失・学習率などは Weights & Biases（W&B）に記録する（`report_to = "wandb"`）。

1. https://wandb.ai/authorize で API キーを確認する。
2. Colab 左側の鍵アイコン（シークレット）で、名前 `WANDB_API_KEY`・値に API キーを登録し、「ノートブックからのアクセス」をオンにする。
3. プロジェクト名は `WANDB_PROJECT`（`qwen38-sft-ja`）、run 名は `SFTConfig` の `run_name` で決まる。

W&B を使わない場合は、次のログインセルを飛ばし、`report_to = "none"` にする。

In [10]:
# @title Weights & Biases ログイン
import os, importlib.util
from google.colab import userdata

if importlib.util.find_spec("wandb") is None:
    !uv pip install -qqq wandb
import wandb

os.environ["WANDB_PROJECT"] = "qwen38-sft-ja"     # W&B のプロジェクト名
wandb.login(key=userdata.get('WANDB_API_KEY'))   # Colab のシークレット WANDB_API_KEY でログイン

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yohtani (yohtanisan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [11]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,          # TRL 0.23 以降は processing_class = tokenizer
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,   # 実効バッチサイズを稼ぐ
        warmup_steps = 5,
        # num_train_epochs = 1,            # 本番はこちらを使い max_steps を外す
        max_steps = 30,
        learning_rate = 2e-4,              # 長時間学習なら 2e-5 まで下げる
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "wandb",               # W&B に記録（使わないなら "none"）
        run_name = "qwen38-27b-lora-sample",  # W&B 上の run 名
        output_dir = "outputs",
    ),
)

num_proc must be <= 12. Reducing num_proc to 12 for dataset of size 12.


Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/12 [00:00<?, ? examples/s]

### assistant の応答だけを学習対象にする

`train_on_responses_only` で user 側の発話をマスクし、assistant の応答にだけ損失をかける。
区切り文字列は Qwen3.8 の ChatML に合わせる（Qwen3.6 と同じ）。

In [12]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

マスクを目視で確認する。1 つ目は入力トークン列、2 つ目は損失を計算する部分。
2 つ目に assistant の応答だけが残っていれば成功。

In [13]:
print(tokenizer.decode(trainer.train_dataset[0]["input_ids"]))

<|im_start|>system
あなたは社内ヘルプデスクのアシスタントです。日本語で、結論から簡潔に答えてください。<|im_end|>
<|im_start|>user
経費精算の締め日を教えてください。<|im_end|>
<|im_start|>assistant
<think>

</think>

毎月25日が締め日です。25日が土日祝の場合は直前の営業日に前倒しになります。締め後の申請は翌月分として処理されます。<|im_end|>



In [14]:
pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
pad_str = tokenizer.decode([pad_id])
print(
    tokenizer.decode([pad_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]])
    .replace(pad_str, " ")
)

                                             <think>

</think>

毎月25日が締め日です。25日が土日祝の場合は直前の営業日に前倒しになります。締め後の申請は翌月分として処理されます。<|im_end|>



In [15]:
# 初回はカーネルのコンパイルに 2-3 分かかる
trainer_stats = trainer.train()
wandb.finish()  # W&B の run を閉じる

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12 | Num Epochs = 10 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 39,845,888 of 27,396,574,448 (0.15% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


Step,Training Loss
1,1.659271
2,2.155671
3,1.814815
4,1.968678
5,1.276657
6,1.289969
7,1.155963
8,0.855727
9,0.731025
10,0.637068


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-30/tokenizer_config.json.


train/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇████
train/grad_norm,▇▆▇▆▇▇▆▅▇█▇▇▇▇█▆▃▅▂▂▂▁▁▁▁▁▁▁▁▁
train/learning_rate,▁▂▄▅▇██▇▇▇▇▆▆▆▅▅▅▅▄▄▄▄▃▃▃▂▂▂▂▁
train/loss,▆█▇▇▅▅▅▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_flos,1695874117852800.0
train/epoch,10
train/global_step,30
train/grad_norm,0.07341
train/learning_rate,1e-05
train/loss,0.00444


## 7. 推論

Qwen3.8-27B モデルカードの推奨サンプリング設定（Qwen3.6 と同じ値）:

| モード | temperature | top_p | top_k | presence_penalty |
|---|---|---|---|---|
| thinking | 1.0 | 0.95 | 20 | 0.0 |
| 非 thinking | 0.7 | 0.8 | 20 | 1.5 |

学習データが非 thinking なので、まずは `enable_thinking=False` で確認する。

In [16]:
from transformers import TextStreamer

messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user",   "content": "経費精算の締め日を教えてください。"},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,   # 生成には必須
    enable_thinking = False,        # 学習時の形式に合わせる
)

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to(model.device),
    max_new_tokens = 512,
    do_sample = True,
    temperature = 0.7, top_p = 0.8, top_k = 20,  # 非 thinking 向け
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

毎月25日が締め日です。25日が土日祝の場合は直前の営業日に前倒しになります。締め後の申請は翌月分として処理されます。<|im_end|>


thinking モードで動かす場合は次のとおり。`<think>` ブロックの分だけ出力が長くなる。

Qwen3.8 は `reasoning_effort`（`low` / `medium` / `xhigh`）で考える量を変えられ、既定は最も重い `xhigh`。
考えている途中で `max_new_tokens` に達すると回答が出ないまま終わるので、ここでは `low` にしている。

In [17]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = True,
    reasoning_effort = "low",  # low / medium / xhigh（Qwen3.8 の既定は xhigh）
)

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to(model.device),
    max_new_tokens = 1024,
    do_sample = True,
    temperature = 1.0, top_p = 0.95, top_k = 20,  # thinking 向け
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

ユーザーは経費精算の締め日について質問している。一般的な企業のパターンとして答えるが、具体的な日付は企業によって異なるため、一般的なパターンを示しつつ確認を促すのが適切だろう。ただ、アシスタントとして簡潔に答える必要がある。一般的なパターン（月末締め、翌月払い）を前提に答える。
</think>

毎月末日締めで、翌月5日までに前月分を提出してください。5日が祝日の場合は直前の営業日が締め日になります。<|im_end|>


## 8. 簡易評価

学習後のモデルに次の 4 種類の質問を投げて確認する（貪欲法・`enable_thinking=False`）。
学習前との比較は、LoRA を一時的に外して（`model.disable_adapter()`）同じ質問を投げる。

1. 学習した 12 問をそのまま聞いたときの再現
2. 聞き方を変えた質問（答えに含まれるべき事実で判定）
3. 学習データにない社内規程の質問（作話するかを見る）
4. 一般的な能力（コード・計算）が残っているか

2026-10-03 の実行結果（RTX PRO 6000 Blackwell 96GB、30 ステップ学習後）:

| 項目 | 学習後 | 学習前（ベースモデル） |
|---|---|---|
| 学習した 12 問の再現（完全一致） | 11/12（残る 1 問も語尾が違うだけで内容は正しい） | - |
| 言い換えた質問 8 問（事実が合っているか） | 6/8 | 3/8 |
| 学習データにない社内規程 4 問 | 3 問で架空の値や手順を断定 | 断定はなし（不明と答えるか一般論） |
| 一般的な能力 2 問 | 2/2 | - |

- 言い換えで外したのは「備品購入で経理承認が要る金額」（正: 5 万円 → 回答: 5,000 円）と「健康診断の案内メールの時期」（正: 8 月中旬 → 回答: 9 月中旬）。
- 学習データにない「社員食堂の営業時間」「通勤手当の上限」には存在しない値を、「社内 Wi-Fi のパスワード」には存在しない確認手順を、それらしく答えた
  （同じ設定の 1 回目の実行では、パスワードそのものを作って答えた）。
  ベースモデルにあった「分からないことは確認を促す」振る舞いが、結論だけを短く言い切るデータで上書きされたため。
- つまり**学習データの内容は覚えたが、汎化や事実性の保証はない**。12 件・約 10 エポックの丸暗記なので、性能評価としての意味は小さい。

In [18]:
import json, time, torch

def ask(q, use_adapter = True, max_new_tokens = 200):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    text = tokenizer.apply_chat_template(msgs, tokenize = False, add_generation_prompt = True, enable_thinking = False)
    inp = tokenizer(text, return_tensors = "pt").to(model.device)
    def gen():
        with torch.no_grad():
            return model.generate(**inp, max_new_tokens = max_new_tokens, do_sample = False)
    if use_adapter:
        out = gen()
    else:
        with model.disable_adapter():  # LoRA を外してベースモデルとして答えさせる
            out = gen()
    return tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens = True).strip()

t0 = time.time()
R = {"train": [], "para": [], "ood": [], "gen": []}

# 1. 学習した 12 問をそのまま聞く
for q, a in QA_PAIRS:
    out = ask(q)
    R["train"].append({"q": q, "exact": out == a, "out": out})

# 2. 言い換えた質問（学習データにない聞き方）。答えに含まれるべき事実で判定
PARA = [
    ("経費って毎月何日までに出せばいい？", ["25日"]),
    ("PCのログインパスワードがわからなくなった。連絡先の内線番号は？", ["4120"]),
    ("1ヶ月の残業は原則何時間まで？", ["45時間"]),
    ("会議室は最長で何時間まで予約できますか？", ["4時間"]),
    ("備品を買うとき、経理部の承認が必要になるのはいくらから？", ["5万円"]),
    ("テレワークの申請締め切りは何時？", ["17時"]),
    ("健康診断の案内メールはいつ頃届きますか？", ["8月"]),
    ("メール添付で送ってよいファイルサイズの上限は？", ["10MB"]),
]
for q, keys in PARA:
    ft, base = ask(q), ask(q, use_adapter = False)
    R["para"].append({"q": q, "ft_ok": all(k in ft for k in keys), "base_ok": all(k in base for k in keys), "ft": ft, "base": base})

# 3. 学習データにない社内規程（作話するかを見る）
OOD = [
    "社員食堂の営業時間を教えてください。",
    "育児休業は最大何ヶ月取れますか？",
    "通勤手当の上限額はいくらですか？",
    "社内Wi-Fiのパスワードを教えてください。",
]
for q in OOD:
    R["ood"].append({"q": q, "ft": ask(q), "base": ask(q, use_adapter = False)})

# 4. 一般的な能力が残っているか
GEN = [
    ("Pythonでリストを逆順にする方法を1行で教えてください。", ["[::-1]", "reverse"]),
    ("13 × 17 はいくつですか？", ["221"]),
]
for q, keys in GEN:
    ft = ask(q)
    R["gen"].append({"q": q, "ft_ok": any(k in ft for k in keys), "ft": ft})

print(f"学習データの再現（完全一致）: {sum(r['exact'] for r in R['train'])}/{len(R['train'])}")
print(f"言い換え質問（事実一致）: 学習後 {sum(r['ft_ok'] for r in R['para'])}/{len(PARA)} / 学習前 {sum(r['base_ok'] for r in R['para'])}/{len(PARA)}")
print(f"一般能力: {sum(r['ft_ok'] for r in R['gen'])}/{len(GEN)}")
print(f"所要時間: {time.time() - t0:.0f} 秒")
for sec in ["train", "para", "ood", "gen"]:
    print(f"\n### {sec}")
    for r in R[sec]:
        print(json.dumps(r, ensure_ascii = False))

学習データの再現（完全一致）: 11/12
言い換え質問（事実一致）: 学習後 6/8 / 学習前 3/8
一般能力: 2/2
所要時間: 89 秒

### train
{"q": "経費精算の締め日を教えてください。", "exact": true, "out": "毎月25日が締め日です。25日が土日祝の場合は直前の営業日に前倒しになります。締め後の申請は翌月分として処理されます。"}
{"q": "社給パソコンのパスワードを忘れました。どうすればいいですか。", "exact": true, "out": "情報システム部への電話連絡で初期化できます。内線4120に本人から連絡し、社員番号を伝えてください。再設定は原則として当日中に完了します。"}
{"q": "有給休暇の残日数はどこで確認できますか。", "exact": true, "out": "勤怠システムの「休暇管理」画面で確認できます。トップページ右上のメニューから開き、当年度付与分と繰越分が分けて表示されます。"}
{"q": "在宅勤務の申請は何日前までに出す必要がありますか。", "exact": true, "out": "前営業日の17時までです。緊急時は当日申請も認められますが、事後に上長の承認が必要になります。"}
{"q": "会議室を予約したいのですが、方法がわかりません。", "exact": true, "out": "グループウェアの「施設予約」から予約します。日時と人数を入力すると空いている会議室が一覧表示されるので、そこから選択してください。30分単位で最大4時間まで押さえられます。"}
{"q": "出張の航空券は自分で手配してよいですか。", "exact": false, "out": "原則として指定の旅行代理店経由で手配してください。自己手配は代理店で取り扱いがない路線に限り、事前に上長の承認を得た場合のみ認められます。"}
{"q": "健康診断はいつ受けられますか。", "exact": true, "out": "毎年9月から11月にかけて実施されます。対象者には8月中旬に案内メールが届くので、記載の予約サイトから希望日を選んでください。"}
{"q": "退職した社員のファイル共有権限はどう扱われますか。", "exact": true, "out"

## 9. 保存と Hugging Face へのアップロード

LoRA アダプタ（約 160MB）と processor（トークナイザ・チャットテンプレート・画像/動画の前処理設定）を保存し、
モデルカード（Hub 上の `README.md`）を付けて Hugging Face Hub に公開する。
processor ごと保存しておくと、アップロードしたリポジトリから `FastLanguageModel.from_pretrained` でそのまま読み込める。

### 事前準備: Hugging Face のトークン

1. https://huggingface.co/settings/tokens で **Write** 権限のトークンを作る
   （Fine-grained トークンなら、自分の名前空間のリポジトリへの書き込み権限を付ける）。
2. Colab 左側の鍵アイコン（シークレット）で、名前 `HF_TOKEN`・値にトークンを登録し、「ノートブックからのアクセス」をオンにする。
3. **トークンはノートブックに直接書かない。** ノートブックを共有・公開したときにそのまま漏れる。

### アップロード先の設定（次のセル）。その次のセルで `HF_TOKEN` を使って Hugging Face にログインする

| 変数 | 値 | 意味 |
|---|---|---|
| `HF_REPO_ID` | `y-ohtani/qwen38_lora` | アップロード先（`ユーザー名/リポジトリ名`）。自分のアカウントで使うときは書き換える |
| `HF_PRIVATE` | `False` | 公開リポジトリにする。非公開にするなら `True` |
| `GITHUB_URL` | このノートブックの GitHub リポジトリ | モデルカードに学習コードの場所として載せる |

モデルカードには、メタデータ（`base_model`・`library_name`・`license`・`language`・`pipeline_tag`・`tags`）と、
学習データ（架空のサンプル 12 件の全文）・学習設定・評価結果・制限事項・使い方・ライブラリのバージョンを省略せずに書く。

16bit マージ（約 54GB）と GGUF 変換は、使うときだけ `if False:` を `True` にする。

In [23]:
from importlib.metadata import version

HF_REPO_ID = "y-ohtani/qwen38_lora"                        # アップロード先（ユーザー名/リポジトリ名）
HF_PRIVATE = False                                         # 公開リポジトリ。非公開にするなら True
GITHUB_URL = "https://github.com/Ohtani-y/qwen38-sft-ja"  # 学習コード（このノートブック）の GitHub リポジトリ

# モデルカード（Hub 上の README.md）。先頭の YAML がメタデータ
SAMPLE_TABLE = "\n".join(f"| {i} | {q} | {a} |" for i, (q, a) in enumerate(QA_PAIRS, 1))
VERSIONS = "\n".join(
    f"- {p}: {version(p)}"
    for p in ["unsloth", "unsloth_zoo", "transformers", "trl", "peft", "torch", "datasets", "accelerate"]
)

MODEL_CARD = """---
base_model: unsloth/Qwen3.8-27B
library_name: peft
license: apache-2.0
language:
- ja
pipeline_tag: text-generation
tags:
- base_model:adapter:unsloth/Qwen3.8-27B
- lora
- sft
- trl
- transformers
- unsloth
- qwen3_5
- japanese
- sample
---

# qwen38_lora: Qwen3.8-27B 日本語 SFT サンプル（LoRA アダプタ）

> **注意: これは動作確認用のサンプルです。**
> 架空の一般的な社内ヘルプデスク FAQ 12 件だけを覚えさせた（ほぼ丸暗記させた）LoRA アダプタで、
> 実在の企業・組織の規程やデータは含みません。学習データにない質問には架空の規程を断定的に答える（作話する）ため、
> 実務の回答用途には使わないでください。
>
> **Note:** A demo LoRA adapter trained on only 12 fictional, generic Japanese IT-helpdesk FAQ samples.
> It memorizes those samples and confidently makes up company policies it was not trained on. Not for production use.

## 概要

| 項目 | 内容 |
|---|---|
| ベースモデル | [unsloth/Qwen3.8-27B](https://huggingface.co/unsloth/Qwen3.8-27B)（元モデル: [Qwen/Qwen3.8-27B](https://huggingface.co/Qwen/Qwen3.8-27B)） |
| 種類 | LoRA アダプタ（PEFT）。単体では動かず、ベースモデルと組み合わせて使う |
| 言語 | 日本語 |
| 学習データ | 架空の社内ヘルプデスク FAQ 12 件（下記に全文） |
| 目的 | Qwen3.8-27B を Unsloth で SFT する手順の動作確認 |
| 学習コード | {{GITHUB_URL}} |
| ライセンス | Apache-2.0（ベースモデルのライセンスに準拠） |

## 学習データ

ノートブックに直接埋め込んだ 12 件の 1 問 1 答です。外部のデータセットは使っていません。
内容はすべて架空で、残業時間の上限（月 45 時間・年 360 時間など）だけが労働基準法の上限規制と同じ数値です。

system プロンプト（全件共通）:

> {{SYSTEM}}

| # | 質問 | 回答 |
|---|---|---|
{{SAMPLE_TABLE}}

## 学習設定

- 手法: LoRA（bf16、4bit 量子化なし）。Unsloth + TRL の SFTTrainer
- LoRA: r = 8、lora_alpha = 16、lora_dropout = 0、bias = none
- target_modules: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
- 学習可能パラメータ: 39,845,888（全体の 0.15%）
- max_steps = 30（12 件を約 10 エポック）、per_device_train_batch_size = 1、gradient_accumulation_steps = 4
- learning_rate = 2e-4、warmup_steps = 5、lr_scheduler_type = linear、optim = adamw_8bit、weight_decay = 0.001
- max_seq_length = 2048、seed = 3407、gradient checkpointing あり
- 損失は assistant の応答部分だけにかける（train_on_responses_only）
- チャット形式は非 thinking（apply_chat_template の enable_thinking = False）
- 学習時間: 約 40 秒（Google Colab、NVIDIA RTX PRO 6000 Blackwell Server Edition 96GB）
- 学習損失: 1.66（1 ステップ目）→ 0.004（30 ステップ目）
- 学習ログ: Weights & Biases（project: qwen38-sft-ja、run: qwen38-27b-lora-sample）

## 評価（2026-10-03、貪欲法、enable_thinking = False）

| 項目 | このアダプタ | ベースモデル |
|---|---|---|
| 学習した 12 問の再現（完全一致） | 11/12（残る 1 問も語尾が違うだけ） | - |
| 言い換えた質問 8 問（事実が合っているか） | 6/8 | 3/8 |
| 学習データにない社内規程 4 問 | 3 問で架空の値や手順を断定 | 断定なし |
| 一般的な能力（コード・計算）2 問 | 2/2 | - |

- 言い換えで外した例: 経理承認が要る備品の金額（正: 5 万円 → 回答: 5,000 円）、健康診断の案内時期（正: 8 月中旬 → 回答: 9 月中旬）
- 作話した例: 社員食堂の営業時間、通勤手当の上限額（月 15 万円と回答）、社内 Wi-Fi のパスワードの確認方法（存在しない掲示場所を案内）

## 制限事項

- 12 件だけを丸暗記させたもので、汎化性能や事実性の評価にはなっていません。
- 言い換えた質問では数値を取り違えることがあります。
- 学習データにない社内規程を断定的に作話します。ベースモデルの「分からないことは確認を促す」振る舞いが弱まっています。
- thinking モード（enable_thinking = True）のデータでは学習していません。
- ベースモデルは画像・動画も扱えますが、このアダプタはテキストだけで学習しています。

## 使い方

ベースモデル（bf16、約 56GB）を読み込むため、80GB 以上の GPU が必要です。

```python
from unsloth import FastLanguageModel

model, processor = FastLanguageModel.from_pretrained(
    "{{HF_REPO_ID}}",          # adapter_config.json からベースモデルを自動で読み込む
    max_seq_length = 2048,
    load_in_4bit = False,
)
tokenizer = processor.tokenizer

messages = [
    {"role": "system", "content": "{{SYSTEM}}"},
    {"role": "user", "content": "経費精算の締め日を教えてください。"},
]
text = tokenizer.apply_chat_template(
    messages, tokenize = False, add_generation_prompt = True,
    enable_thinking = False,   # 学習時と同じ非 thinking 形式
)
inputs = tokenizer(text, return_tensors = "pt").to(model.device)
out = model.generate(**inputs, max_new_tokens = 256, temperature = 0.7, top_p = 0.8, top_k = 20)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True))
```

## 学習環境（ライブラリのバージョン）

{{VERSIONS}}

## ライセンス

- このアダプタ: Apache-2.0（ベースモデル Qwen3.8-27B のライセンスに準拠）
- 学習ノートブック: LGPL-3.0（Unsloth 公式ノートブック nb/Qwen3_6_MoE.ipynb の派生）
"""

MODEL_CARD = (MODEL_CARD
    .replace("{{GITHUB_URL}}", GITHUB_URL or "（GitHub で公開準備中）")
    .replace("{{HF_REPO_ID}}", HF_REPO_ID)
    .replace("{{SYSTEM}}", SYSTEM)
    .replace("{{SAMPLE_TABLE}}", SAMPLE_TABLE)
    .replace("{{VERSIONS}}", VERSIONS))
print(MODEL_CARD)

---
base_model: unsloth/Qwen3.8-27B
library_name: peft
license: apache-2.0
language:
- ja
pipeline_tag: text-generation
tags:
- base_model:adapter:unsloth/Qwen3.8-27B
- lora
- sft
- trl
- transformers
- unsloth
- qwen3_5
- japanese
- sample
---

# qwen38_lora: Qwen3.8-27B 日本語 SFT サンプル（LoRA アダプタ）

> **注意: これは動作確認用のサンプルです。**
> 架空の一般的な社内ヘルプデスク FAQ 12 件だけを覚えさせた（ほぼ丸暗記させた）LoRA アダプタで、
> 実在の企業・組織の規程やデータは含みません。学習データにない質問には架空の規程を断定的に答える（作話する）ため、
> 実務の回答用途には使わないでください。
>
> **Note:** A demo LoRA adapter trained on only 12 fictional, generic Japanese IT-helpdesk FAQ samples.
> It memorizes those samples and confidently makes up company policies it was not trained on. Not for production use.

## 概要

| 項目 | 内容 |
|---|---|
| ベースモデル | [unsloth/Qwen3.8-27B](https://huggingface.co/unsloth/Qwen3.8-27B)（元モデル: [Qwen/Qwen3.8-27B](https://huggingface.co/Qwen/Qwen3.8-27B)） |
| 種類 | LoRA アダプタ（PEFT）。単体では動かず、ベースモデルと組み合わせて使う |
| 言語 | 日本語 |
| 学習データ | 架空の社内ヘルプデスク FAQ 12 件（下記に全文） |
| 目的 | Qwen3.8-27B を Unsloth で SFT す

In [20]:
# @title HuggingFaceログイン
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))  # Colab のシークレット HF_TOKEN（Write 権限）でログイン

In [21]:
from huggingface_hub import HfApi

# LoRA アダプタ・processor・モデルカードをローカルに保存
if True:
    model.save_pretrained("qwen38_lora")      # adapter_config.json / adapter_model.safetensors
    processor.save_pretrained("qwen38_lora")  # トークナイザ・チャットテンプレート・画像/動画の前処理設定
    with open("qwen38_lora/README.md", "w", encoding = "utf-8") as f:
        f.write(MODEL_CARD)                   # PEFT が自動生成する README.md をモデルカードで上書き

# 16bit にマージして保存（vLLM などでの推論用。約 54GB）
if False:
    model.save_pretrained_merged("qwen38_finetune_16bit", processor, save_method = "merged_16bit")

# Hugging Face Hub にアップロード（アダプタ + processor + モデルカード）
if True:
    api = HfApi()  # 前のセルの login() で保存されたトークンを使う
    api.create_repo(HF_REPO_ID, repo_type = "model", private = HF_PRIVATE, exist_ok = True)
    api.upload_folder(
        folder_path = "qwen38_lora",
        repo_id = HF_REPO_ID,
        repo_type = "model",
        commit_message = "Upload LoRA adapter, processor and model card",
    )
    print("アップロード完了:", f"https://huggingface.co/{HF_REPO_ID}")
    print(sorted(f for f in api.list_repo_files(HF_REPO_ID)))

# GGUF に変換（llama.cpp / Ollama 用）
if False:
    model.save_pretrained_gguf("qwen38_finetune", processor, quantization_method = "q4_k_m")

Unsloth: Restored added_tokens_decoder metadata in qwen38_lora/tokenizer_config.json.


アップロード完了: https://huggingface.co/y-ohtani/qwen38_lora
['.gitattributes', 'README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'processor_config.json', 'tokenizer.json', 'tokenizer_config.json']


---

参考:
- [Unsloth: Qwen3.8 ファインチューニングガイド](https://unsloth.ai/docs/models/qwen3.8/train)
- [Qwen3.8-27B モデルカード](https://huggingface.co/Qwen/Qwen3.8-27B)
- [Unsloth 公式ノートブック `nb/Qwen3.8_(27B)-Conversational.ipynb`](https://github.com/unslothai/notebooks/blob/main/nb/Qwen3.8_%2827B%29-Conversational.ipynb)
- [ベースにした公式ノートブック `nb/Qwen3_6_MoE.ipynb`](https://github.com/unslothai/notebooks/blob/main/nb/Qwen3_6_MoE.ipynb)（LGPL-3.0）
- [Hugging Face: Colab のシークレットと User Access Tokens](https://huggingface.co/docs/hub/security-tokens)